# Project 3 : AI Recommendation Logic
# Capstone : Tech Stack Reommender
# Approach : Content-Based Filtering using TF_IDF + Cosine Similarity

### Importing Libraries

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Building The Job Role DataSet

##### Each "item" is a job role, described by its required skills.
#####In a real project this would be loaded from raw_skills.csv,
##### but it's defined inline here so the notebook runs standalone.
##### You can swap this block for: raw_skills_df = pd.read_csv("raw_skills.csv")

In [6]:
raw_skills_data = {
    "Job_role" : [
        "Data Scientist",
        "Machine Learning Engineer",
        "Data Analyst" ,
        "Frontend Developer",
        "Full Stack Developer" ,
        "Backend Developer" ,
        "DevOps Engineer" ,
        "Cloud Architect" ,
        "Site Reliability Engineer",
        "Mobile App Developer"
    ] ,
"Skills" : [
    "Python SQL Machine Learning Data Analyst Statistics Pandas " ,
    "Python Tensorflow PyTorch Machine Learning Deep Learning Data Analysis ",
    "SQL Excel Data Anaylsis statistics Python Visualization ",
    "JavaScript React CSS HTML UI UX Web Design " ,
    "Javascript Python React Node APIs Databases Web Design " ,
    "Java Python SQL APIs REST Databases Backend " ,
    "AWS Docker Kubernetes CI CD Automation Linux Cloud " ,
    "AWS Cloud Computing Azure Networking Automation Kubernetes ",
    "Linux Automation Kubernetes Cloud Monitoring CI CD " ,
    "Java Kotlin Swift Mobile UI UX APIs " ]
}

raw_skills_data = pd.DataFrame(raw_skills_data)


In [7]:
raw_skills_data.head()

,Job_role,Skills
0,Data Scientist,Python SQL Machine Learning Data Analyst Stati...
1,Machine Learning Engineer,Python Tensorflow PyTorch Machine Learning Dee...
2,Data Analyst,SQL Excel Data Anaylsis statistics Python Visu...
3,Frontend Developer,JavaScript React CSS HTML UI UX Web Design
4,Full Stack Developer,Javascript Python React Node APIs Databases We...


### Capture User Input

##### Specfic requires a Minimum of 3 User Input
##### These represent the users a raw skills / Interests


In [8]:
user_skills = ["Python" , "Cloud Computing" , "Automation"]

assert len(user_skills) >=3 , "Minimum of 3 user inputs required. "


In [9]:
# Convert the list of skills into a single space-separated string
# so it can be vectorized the same way as the job role skill string
user_profile_text = " ".join(user_skills)
print("\n User Profile Input:" , user_profile_text)


 User Profile Input: Python Cloud Computing Automation


### Vector Mapping : TF-IDF
##### we fit the TF-IDF Vectorizer on job roles + the user profile Together
##### so that both sides map into the exact same vocabulary space

In [10]:
corpus = raw_skills_data["Skills"].tolist() + [user_profile_text]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# The last row of the matrix is the user's vector;
# all the rpws before it are the job role vectors.

job_vectors = tfidf_matrix[:-1]
user_vector = tfidf_matrix[-1]

print("\n Vocabulary Learned by TF-IDF:")
print(vectorizer.get_feature_names_out())


 Vocabulary Learned by TF-IDF:
['analysis' 'analyst' 'anaylsis' 'apis' 'automation' 'aws' 'azure'
 'backend' 'cd' 'ci' 'cloud' 'computing' 'css' 'data' 'databases' 'deep'
 'design' 'docker' 'excel' 'html' 'java' 'javascript' 'kotlin'
 'kubernetes' 'learning' 'linux' 'machine' 'mobile' 'monitoring'
 'networking' 'node' 'pandas' 'python' 'pytorch' 'react' 'rest' 'sql'
 'statistics' 'swift' 'tensorflow' 'ui' 'ux' 'visualization' 'web']


### Cosine Similarity
###### Compare the user vector against every job role vector.
#####  Cosine similarity is used (not Euclidean distance) because it is
#####  invariant to vector magnitude -- it measures orientation, not length.

In [12]:
similarity_scores = cosine_similarity(user_vector , job_vectors).flatten()

raw_skills_data["Similarity_Score"] = similarity_scores
print("\n Scored Job Roles: ")
print(raw_skills_data[["Job_role" , "Similarity_Score"]])


 Scored Job Roles: 
                    Job_role  Similarity_Score
0             Data Scientist          0.092055
1  Machine Learning Engineer          0.074794
2               Data Analyst          0.096077
3         Frontend Developer          0.000000
4       Full Stack Developer          0.092916
5          Backend Developer          0.098685
6            DevOps Engineer          0.279329
7            Cloud Architect          0.528556
8  Site Reliability Engineer          0.300431
9       Mobile App Developer          0.000000


### Rank By Similarity

In [15]:
ranked_df = raw_skills_data.sort_values(by="Similarity_Score" , ascending=False ).reset_index(drop=True)
print("\n Ranked Job Roles (highest match first) :")
print(ranked_df[["Job_role" , "Similarity_Score"]])


 Ranked Job Roles (highest match first) :
                    Job_role  Similarity_Score
0            Cloud Architect          0.528556
1  Site Reliability Engineer          0.300431
2            DevOps Engineer          0.279329
3          Backend Developer          0.098685
4               Data Analyst          0.096077
5       Full Stack Developer          0.092916
6             Data Scientist          0.092055
7  Machine Learning Engineer          0.074794
8         Frontend Developer          0.000000
9       Mobile App Developer          0.000000


### Top_N Output (Top 3)

In [18]:
Top_N = 3
top_recommendations = ranked_df.head(Top_N)

print(f"\n === Top {Top_N} Recommended Career Paths === ")
for i , row in top_recommendations.iterrows():
  print(f"{i+1} . {row['Job_role']}  (Match Score: {row['Similarity_Score']}:.2f)")


 === Top 3 Recommended Career Paths === 
1 . Cloud Architect  (Match Score: 0.5285562312678883:.2f)
2 . Site Reliability Engineer  (Match Score: 0.3004313230727787:.2f)
3 . DevOps Engineer  (Match Score: 0.27932888810390427:.2f)


#Optional
### Cold Start Awareness Demo
##### Demonstrates what happens with an empty / unseen-vocabulary profile


In [21]:
cold_start_user = [""] #a brand-new user with no skills entered
cold_corpus = raw_skills_data["Skills"].tolist() + cold_start_user
cold_tfidf = vectorizer.transform(cold_start_user) # reuse the fitted vocabulary


cold_scores = cosine_similarity(cold_tfidf, job_vectors).flatten()
print("\n Cold Start Example - similarity scores with no input :" , cold_scores)
print("(All zeros : this demonstrates the 'Cold Start Probelm ' described in the deck.)")
print("Real Systems handle this using onboarding surverys , trending fallbacks , or metadata inference. ")


 Cold Start Example - similarity scores with no input : [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(All zeros : this demonstrates the 'Cold Start Probelm ' described in the deck.)
Real Systems handle this using onboarding surverys , trending fallbacks , or metadata inference. 


# Building a Recommendation System

In [24]:
def top_recommendations(user_input_skills, top_n=3):
    # Ensure user_input_skills is not empty
    if not user_input_skills:
        return pd.DataFrame() # Return empty DataFrame if no skills

    user_profile_text = " ".join(user_input_skills)

    # Use the globally defined vectorizer to transform the new user skills
    user_vector = vectorizer.transform([user_profile_text])

    # Calculate similarity scores
    similarity_scores = cosine_similarity(user_vector, job_vectors).flatten()

    # Create a copy of raw_skills_data to add similarity scores for this specific user
    recommendations_df = raw_skills_data.copy()
    recommendations_df["Similarity_Score"] = similarity_scores

    # Rank and get top_n
    ranked_recommendations = recommendations_df.sort_values(
        by="Similarity_Score", ascending=False
    ).reset_index(drop=True)

    return ranked_recommendations.head(top_n)

def run_recommender():
    print("=" * 55)
    print("   AI TECH STACK RECOMMENDER")
    print("=" * 55)
    print("Enter your skills separated by commas (minimum 3).")
    print("Example: Python, Cloud Computing, Automation")
    print("Type 'exit' anytime to quit.\n")

    while True:
        raw_input_text = input("Your skills: ")

        # Exit option
        if raw_input_text.strip().lower() == "exit":
            print("Goodbye! 👋")
            break

        # Convert comma-separated input into a clean list of skills
        user_skills = [skill.strip() for skill in raw_input_text.split(",") if skill.strip()]

        # Validate minimum input requirement
        if len(user_skills) < 3:
            print("⚠️  Please enter at least 3 skills, separated by commas.\n")
            continue

        # Get recommendations
        top_matches = top_recommendations(user_skills, top_n=3)

        # Display results
        print(f"\nBased on your skills ({', '.join(user_skills)}), here are your Top 3 job matches:\n")
        for i, row in top_matches.iterrows():
            print(f"{i + 1}. {row['Job_role']}  —  Match Score: {row['Similarity_Score']:.2f}")
        print()  # blank line for readability


# ---------------------------------------------------------
# 5. RUN THE SYSTEM
# ---------------------------------------------------------
if __name__ == "__main__":
    run_recommender()

   AI TECH STACK RECOMMENDER
Enter your skills separated by commas (minimum 3).
Example: Python, Cloud Computing, Automation
Type 'exit' anytime to quit.

Your skills: Python , Excel , SQL

Based on your skills (Python, Excel, SQL), here are your Top 3 job matches:

1. Data Analyst  —  Match Score: 0.60
2. Backend Developer  —  Match Score: 0.29
3. Data Scientist  —  Match Score: 0.27

Your skills: exit
Goodbye! 👋
